# AgroPest-12 Part B -- YOLOv12-m COCO Baseline (20% Labels): Training, Evaluation & Error Analysis

**Student:** Md. Sadik Shahriar  
**Student ID:** 2023-2-60-103  
**Model:** YOLOv12-medium (`yolo12m.pt` -- COCO-pretrained, 20.2M params, 67.5 GFLOPs)  
**Dataset:** AgroPest-12-PartB-Safe -- `finetune_20` 2,295 images, valid 1,089, test 543  
**Course:** CSE 445 -- Computer Vision | Assignment 1 (Part B comparison)
**Platform:** Kaggle -- 2x Tesla T4 GPUs (15 GB VRAM each)  
**Mirror of:** Notebook 3 (Part A full-data run) run on the Part B 20% fine-tune split

---

This is the **COCO-pretrained baseline** for Assignment Part B. It fine-tunes YOLOv12-m on
the exact same 20% label budget (`finetune_20`) used by every downstream SSL arm
(`partb-01b/02b/03b`), under the **identical frozen protocol** (50 epochs, batch=32,
640x640, same augmentation, seed=445). The val/test numbers here are the reference point
for the Part B question: did label-free SSL pretraining on the 80% SSL pool actually
improve mAP over plain COCO-pretrained fine-tuning?

The cell layout mirrors Notebook 3 cell-for-cell so the comparisons stay on equal footing.

In [ ]:
# ============================================================
# CONFIG -- YOLOv12-m COCO Baseline (Part B, rho = 0.20)
# Every parameter below must be justifiable in the viva.
# ============================================================
import os

# ---- Dataset (Part B Safe split) ----
PARTB_ROOT = "/kaggle/input/datasets/sadikshahriar/agropest-12-partb-safe/AgroPest-12-PartB-Safe"
DATA_YAML    = "/kaggle/working/data_baseline.yaml"
CLASS_NAMES  = [
    "Ants", "Bees", "Beetles", "Caterpillars",
    "Earthworms", "Earwigs", "Grasshoppers", "Moths",
    "Slugs", "Snails", "Wasps", "Weevils"
]
NUM_CLASSES  = 12

# ---- Model ----
MODEL_NAME   = "yolo12m.pt"      # COCO-pretrained (same file as Notebook 3);
                                 # the SSL arms instead start from yolo12m.yaml
                                 # + SSL-backbone surgery
PROJECT      = "/kaggle/working/agropest_baseline_yolov12"
RUN_NAME     = "yolov12_baseline_coco20"

# ============================================================
# MODEL-SIZE JUSTIFICATION  (unchanged from Notebook 3)
#   s (small):  insufficient capacity for 12 classes
#   m (medium): BALANCED CHOICE -- enough capacity without overfitting;
#               50 epochs x ~1.5 min/epoch ~= 1.3 h on 2xT4 (2,295 images)
#   l (large):  overkill at rho=0.2 -- higher overfit risk on 2,295 images
# ============================================================

# ---- Training hyperparameters (FROZEN -- same as the SSL arms) ----
EPOCHS        = 50
BATCH         = 32              # 16 per GPU on 2xT4 (DDP splits the batch)
IMGSZ         = 640             # native dataset resolution
DEVICE        = [0, 1]          # DDP across both T4 GPUs
AMP           = True            # FP16 halves VRAM, critical for T4
OPTIMIZER     = "auto"          # iterations = ceil(2295/64)*50 ~= 1,800 < 10k
                                 # -> resolves to AdamW (identical to the arms)
LR0           = 0.01            # peak LR; overridden by the AdamW fit LR
LRF           = 0.01            # final LR = lr0 * lrf = 1e-4
MOMENTUM      = 0.937           # SGD-family momentum (AdamW uses betas)
WEIGHT_DECAY  = 0.0005          # L2 regularisation weight
WARMUP_EPOCHS = 3.0             # 3 warmup epochs stabilises early gradients
WARMUP_MOMENTUM = 0.8
WARMUP_BIAS_LR  = 0.1

# ---- Augmentation ----
CLOSE_MOSAIC  = 10              # disable mosaic last 10 epochs -> tighter boxes
MOSAIC        = 1.0             # multi-image context helps rare classes
MIXUP         = 0.15            # mild label mixing flattens class imbalance
DEGREES       = 10.0            # arbitrary field orientation
SCALE         = 0.5             # 0.75x-1.25x natural size variation
SHEAR         = 0.0             # shear distorts insect morphology -- excluded
TRANSLATE     = 0.1             # partially-visible insects at edges
FLIPLR        = 0.5             # no semantic left/right orientation
FLIPUD        = 0.5             # any angle relative to a top-down camera
HSV_H         = 0.01            # REDUCED from default -- colour is key
HSV_S         = 0.4             # REDUCED from 0.7 -- preserves colour classes
HSV_V         = 0.4             # outdoor lighting variation

# ---- Regularisation & Scheduling ----
RECT          = False           # all images square -- no padding
COS_LR        = False           # linear LR decay
PATIENCE      = 0               # NO early stop -> exactly 50 epochs, like 01b
NBS           = 64              # accumulate = round(64/32)=2 -> effective 64
WORKERS       = 4               # 2 dataloader workers per GPU
SEED          = 445             # shared Part B seed

# ---- Inference ----
CONF          = 0.25            # standard eval confidence threshold
IOU           = 0.7             # standard NMS IoU threshold
MAX_DET       = 300             # max detections per image

print("CONFIG loaded -- YOLOv12-m COCO baseline | 50 epochs | batch=32 | 640x640 | 2xT4 DDP")

# Environment Setup

In [ ]:
!pip install ultralytics --no-deps

import torch
import os, yaml, shutil, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
import cv2
from pathlib import Path
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

# ---- GPU check ----
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
ngpu = torch.cuda.device_count()
print(f"GPU count: {ngpu}")
for i in range(ngpu):
    props = torch.cuda.get_device_properties(i)
    # Changed 'total_mem' to 'total_memory' below:
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"{props.total_memory / 1024**3:.1f} GB VRAM")

# ---- Plot styling ----
plt.rcParams.update({
    'figure.dpi': 100,
    'savefig.dpi': 100,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# Distinct colours for 12 classes
CLASS_COLORS = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8",
    "#f58231", "#911eb4", "#46f0f0", "#f032e6",
    "#bcf60c", "#fabebe", "#008080", "#e6beff"
]

print("\nEnvironment ready.")

# Data Load

Generate a `data.yaml` pointing to the **AgroPest-12-PartB-Safe** dataset on Kaggle.

Part B split protocol (fixed seed 445, exclusive train/val/test):
- **finetune_20:** 2,295 labelled images -- fine-tune only
- **valid:**       1,089 labelled images -- validation only (model selection)
- **test:**          543 labelled images -- final evaluation only

The SSL pool (9,180 label-free images) is deliberately NOT attached: a baseline
must never see it.

In [ ]:
from ultralytics import YOLO
import yaml
from pathlib import Path

# ------------------------------------------------------------
# Generate data.yaml with absolute Kaggle paths
# ------------------------------------------------------------

# Verify the Part B dataset directory exists
assert os.path.isdir(PARTB_ROOT), f"PartB dataset not found at {PARTB_ROOT}"
for sub in ["finetune_20/images", "finetune_20/labels",
            "valid/images", "valid/labels",
            "test/images", "test/labels"]:
    p = os.path.join(PARTB_ROOT, sub)
    assert os.path.isdir(p), f"Missing directory: {p}"

# Print split sizes
for split, tag in [("finetune_20", "train"), ("valid", "valid"), ("test", "test")]:
    n_img = len(os.listdir(os.path.join(PARTB_ROOT, split, "images")))
    n_lbl = len(os.listdir(os.path.join(PARTB_ROOT, split, "labels")))
    print(f"  {tag:>6}: {n_img:>5} images | {n_lbl:>5} labels")

# Write data.yaml
data_config = {
    "train": os.path.join(PARTB_ROOT, "finetune_20", "images"),
    "val":   os.path.join(PARTB_ROOT, "valid", "images"),
    "test":  os.path.join(PARTB_ROOT, "test", "images"),
    "nc":    NUM_CLASSES,
    "names": CLASS_NAMES,
}

yaml_path = Path(DATA_YAML)
with open(yaml_path, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

# Load and print to confirm
with open(yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

print(f"\nDataset loaded successfully from: {PARTB_ROOT}")
print("-" * 40)
print(yaml.dump(data_cfg, sort_keys=False))

# Train Model

Fine-tuning YOLOv12-m (COCO-pretrained) for 50 epochs on the 2,295-image
`finetune_20` training split. Validation runs every epoch on the 1,089-image
valid split. The best checkpoint is saved automatically
(fitness = 0.1*mAP50 + 0.9*mAP50-95).

**Frozen protocol -- identical to the SSL arms (01b/02b/03b):**
- 50 epochs, batch=32 (16/GPU), 640x640, seed=445, patience=0 (no early stop)
- optimizer='auto' -> iterations ~= 1,800 < 10k -> AdamW (same as the arms)
- Mosaic off for the last 10 epochs (close_mosaic=10); mixup=0.15; HSV-S=0.4
- Every augmentation value matches Notebook 3 / Part B for comparability

In [ ]:
#Sometimes, when a long training run starts, YOLO pauses in the console and asks you to paste a WandB API key. If you walk away from your computer, the training will just sit there waiting forever, burning your GPU quota.
import os
os.environ["WANDB_MODE"] = "disabled"

In [ ]:
model = YOLO(MODEL_NAME)  # COCO-pretrained baseline

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=DEVICE,
    amp=AMP,
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    warmup_momentum=WARMUP_MOMENTUM,
    warmup_bias_lr=WARMUP_BIAS_LR,
    close_mosaic=CLOSE_MOSAIC,
    mosaic=MOSAIC,
    mixup=MIXUP,
    degrees=DEGREES,
    scale=SCALE,
    shear=SHEAR,
    translate=TRANSLATE,
    fliplr=FLIPLR,
    flipud=FLIPUD,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
    rect=RECT,
    cos_lr=COS_LR,
    patience=PATIENCE,
    nbs=NBS,
    workers=WORKERS,
    seed=SEED,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    val=True,
    plots=True,
    save=True,
)

print("\nTraining complete. Best model saved in:")
best_pt = os.path.join(PROJECT, RUN_NAME, "weights", "best.pt")
print(best_pt)

# Model Performance

Ultralytics auto-generates training curves and validation metrics during
training. The plots below show loss components (box, cls, dfl) and
detection metrics (Precision, Recall, mAP@50, mAP@50:95) across all epochs.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
from IPython.display import Image, display

# --- 1. Define paths and title ---
results_dir = os.path.join(PROJECT, RUN_NAME)
image_filename = "results.png"
image_path = os.path.join(results_dir, image_filename)

my_custom_title = "All-in-One Training & Validation Curves (YOLOv12-m -- Part B rho=0.20)"

# --- 2. Load and display the image with a matplotlib title ---
try:
    img = mpimg.imread(image_path)
    plt.figure(figsize=(12, 10), dpi=150)
    plt.imshow(img)
    plt.title(my_custom_title, fontsize=16)
    plt.axis('off')
    plt.show()
except FileNotFoundError:
    print(f"Error: Could not find '{image_filename}' in {results_dir}")
    print("Please make sure the 'results_dir' path is correct.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# --- 1. Define paths and title ---
results_dir = os.path.join(PROJECT, RUN_NAME)
image_filename = "BoxF1_curve.png"
image_path = os.path.join(results_dir, image_filename)

my_custom_title = "F1-Confidence Curve (YOLOv12-m -- Part B rho=0.20)"

# --- 2. Load and display ---
try:
    img = mpimg.imread(image_path)
    plt.figure(figsize=(10, 8), dpi=150)
    plt.imshow(img)
    plt.title(my_custom_title, fontsize=16)
    plt.axis('off')
    plt.show()
except FileNotFoundError:
    print(f"Error: Could not find '{image_filename}' in {results_dir}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# --- 1. Define paths and title ---
results_dir = os.path.join(PROJECT, RUN_NAME)
image_filename = "BoxPR_curve.png"
image_path = os.path.join(results_dir, image_filename)

my_custom_title = "Precision-Recall Curve (YOLOv12-m -- Part B rho=0.20)"

# --- 2. Load and display ---
try:
    img = mpimg.imread(image_path)
    plt.figure(figsize=(10, 8), dpi=150)
    plt.imshow(img)
    plt.title(my_custom_title, fontsize=16)
    plt.axis('off')
    plt.show()
except FileNotFoundError:
    print(f"Error: Could not find '{image_filename}' in {results_dir}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# --- 1. Define paths and title ---
results_dir = os.path.join(PROJECT, RUN_NAME)
image_filename = "BoxP_curve.png"
image_path = os.path.join(results_dir, image_filename)

my_custom_title = "Precision-Confidence Curve (YOLOv12-m -- Part B rho=0.20)"

# --- 2. Load and display ---
try:
    img = mpimg.imread(image_path)
    plt.figure(figsize=(10, 8), dpi=150)
    plt.imshow(img)
    plt.title(my_custom_title, fontsize=16)
    plt.axis('off')
    plt.show()
except FileNotFoundError:
    print(f"Error: Could not find '{image_filename}' in {results_dir}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# --- 1. Define paths and title ---
results_dir = os.path.join(PROJECT, RUN_NAME)
image_filename = "BoxR_curve.png"
image_path = os.path.join(results_dir, image_filename)

my_custom_title = "Recall-Confidence Curve (YOLOv12-m -- Part B rho=0.20)"

# --- 2. Load and display ---
try:
    img = mpimg.imread(image_path)
    plt.figure(figsize=(10, 8), dpi=150)
    plt.imshow(img)
    plt.title(my_custom_title, fontsize=16)
    plt.axis('off')
    plt.show()
except FileNotFoundError:
    print(f"Error: Could not find '{image_filename}' in {results_dir}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# --- 1. Define paths and title ---
results_dir = os.path.join(PROJECT, RUN_NAME)
image_filename = "confusion_matrix.png"
image_path = os.path.join(results_dir, image_filename)

my_custom_title = "Validation Set -- Confusion Matrix (YOLOv12-m -- Part B rho=0.20)"

# --- 2. Load and display ---
try:
    img = mpimg.imread(image_path)
    plt.figure(figsize=(10, 8), dpi=150)
    plt.imshow(img)
    plt.title(my_custom_title, fontsize=16)
    plt.axis('off')
    plt.show()
except FileNotFoundError:
    print(f"Error: Could not find '{image_filename}' in {results_dir}")
    print("Please make sure the 'results_dir' path is correct.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# --- 1. Define paths and title ---
results_dir = os.path.join(PROJECT, RUN_NAME)
image_filename = "labels.jpg"
image_path = os.path.join(results_dir, image_filename)

my_custom_title = "Dataset Label Statistics (YOLOv12-m -- Part B rho=0.20)"

# --- 2. Load and display ---
try:
    img = mpimg.imread(image_path)
    plt.figure(figsize=(10, 8), dpi=150)
    plt.imshow(img)
    plt.title(my_custom_title, fontsize=16)
    plt.axis('off')
    plt.show()
except FileNotFoundError:
    print(f"Error: Could not find '{image_filename}' in {results_dir}")
    print("Please make sure the 'results_dir' path is correct.")

# Evaluation

Evaluating the best checkpoint on both the validation and test splits.
Metrics reported: Precision, Recall, mAP@50, mAP@50:95, F1-score (mean over
classes). Per-class metrics and inference speed (FPS) are also measured on
the test split. The confusion matrix shows class-level predictions vs ground
truth on the test set.

**For the viva:** these are the COCO-baseline anchor; compare with the
SSL-initialised results of 01b/02b/03b under the same protocol.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO

# ------------------------------------------------------------
# 0. Load the trained model (best.pt)
# ------------------------------------------------------------
best_model_path = os.path.join(PROJECT, RUN_NAME, "weights", "best.pt")
print(f"Loading trained model from:\n{best_model_path}\n")

model = YOLO(best_model_path)

# yaml_path is already defined in the Data Load cell
print(f"Using data yaml: {yaml_path}\n")

# ------------------------------------------------------------
# 1. Validation Set Evaluation
# ------------------------------------------------------------
print("Evaluating on VALIDATION set...")
with np.errstate(invalid='ignore'):
    val_metrics = model.val(data=str(yaml_path), split="val")

val_p  = float(val_metrics.box.p.mean())
val_r  = float(val_metrics.box.r.mean())
val_map50   = float(val_metrics.box.map50.mean())
val_map5095 = float(val_metrics.box.map.mean())
val_f1 = float(val_metrics.box.f1.mean())

print("\nValidation metrics (mean over classes):")
print(f"  Precision:  {val_p:.4f}")
print(f"  Recall:     {val_r:.4f}")
print(f"  mAP@0.50:   {val_map50:.4f}")
print(f"  mAP@0.50:0.95: {val_map5095:.4f}")
print(f"  F1-score:   {val_f1:.4f}")
print("-" * 40)

# ------------------------------------------------------------
# 2. Test Set Evaluation
# ------------------------------------------------------------
print("\nEvaluating on TEST set...")
with np.errstate(invalid='ignore'):
    test_metrics = model.val(data=str(yaml_path), split="test")

test_p  = float(test_metrics.box.p.mean())
test_r  = float(test_metrics.box.r.mean())
test_map50   = float(test_metrics.box.map50.mean())
test_map5095 = float(test_metrics.box.map.mean())
test_f1 = float(test_metrics.box.f1.mean())

print("\nTest metrics (mean over classes):")
print(f"  Precision:  {test_p:.4f}")
print(f"  Recall:     {test_r:.4f}")
print(f"  mAP@0.50:   {test_map50:.4f}")
print(f"  mAP@0.50:0.95: {test_map5095:.4f}")
print(f"  F1-score:   {test_f1:.4f}")
print("-" * 40)

# ------------------------------------------------------------
# 3. Per-Class Metrics Table (Test Set)
# ------------------------------------------------------------
# Removed .cpu().numpy() because Ultralytics already outputs these as NumPy arrays
per_cls_p     = test_metrics.box.p
per_cls_r     = test_metrics.box.r
per_cls_m50   = test_metrics.box.map50
per_cls_m5095 = test_metrics.box.map
per_cls_f1    = test_metrics.box.f1

df_per_class = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Precision":    np.round(per_cls_p,    4),
    "Recall":       np.round(per_cls_r,    4),
    "mAP@50":       np.round(per_cls_m50,  4),
    "mAP@50-95":    np.round(per_cls_m5095, 4),
    "F1":           np.round(per_cls_f1,   4),
})
print("\nPer-Class Metrics (Test Set):")
display(df_per_class)

# ------------------------------------------------------------
# 4. Confusion Matrix from Test Run
# ------------------------------------------------------------
try:
    results_dir = test_metrics.save_dir
    cm_filename = "confusion_matrix.png"
    cm_path = os.path.join(results_dir, cm_filename)

    print(f"\nLoading confusion matrix from:\n{cm_path}\n")
    img = mpimg.imread(cm_path)

    plt.figure(figsize=(8, 6), dpi=150)
    plt.imshow(img)
    plt.title("Test Set -- Confusion Matrix (YOLOv12-m -- Part B rho=0.20)", fontsize=14)
    plt.axis('off')
    plt.show()
except FileNotFoundError:
    print(f"Could not find '{cm_filename}' in {results_dir}")

# ------------------------------------------------------------
# 5. Inference Speed (FPS) -- 100 Test Images
# ------------------------------------------------------------
test_img_dir = os.path.join(PARTB_ROOT, "test", "images")
test_images  = sorted(
    [os.path.join(test_img_dir, f) for f in os.listdir(test_img_dir)
     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
)[:100]

t0 = time.time()
preds = model.predict(
    source=test_images,
    imgsz=IMGSZ,
    device=DEVICE,
    conf=CONF,
    iou=IOU,
    max_det=MAX_DET,
    verbose=False,
)
t_total = time.time() - t0
fps = 100 / t_total
print(f"\n{'─'*40}")
print(f"  Inference speed (100 images): {t_total:.2f} s -> {fps:.1f} FPS")
print(f"  Per-image time: {t_total/100*1000:.1f} ms")

# Model Inference -- Sample Predictions by Class

Run inference using the best checkpoint on the test split. To fairly
assess performance across all 12 pest classes, we select **3 images per
class** (36 images total) from the test set -- each image containing at
least one instance of the target class. Predictions are visualised with
colour-coded bounding boxes and class labels.

The grid below helps spot class-specific failure patterns
(e.g., Slugs vs Snails confusion).

In [ ]:
model = YOLO(best_model_path)

# ------------------------------------------------------------
# 1. Find 3 test images per class by scanning label files
# ------------------------------------------------------------
test_img_dir = os.path.join(PARTB_ROOT, "test", "images")
test_lbl_dir = os.path.join(PARTB_ROOT, "test", "labels")

class_images = {i: [] for i in range(NUM_CLASSES)}
for fname in sorted(os.listdir(test_lbl_dir)):
    if not fname.endswith(".txt"):
        continue
    img_stem = os.path.splitext(fname)[0]
    img_path = None
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = os.path.join(test_img_dir, img_stem + ext)
        if os.path.exists(candidate):
            img_path = candidate
            break
    if img_path is None:
        continue
    lbl_path = os.path.join(test_lbl_dir, fname)
    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cid = int(float(parts[0]))
                if cid in class_images and len(class_images[cid]) < 3:
                    class_images[cid].append((img_path, lbl_path))
    if all(len(v) >= 3 for v in class_images.values()):
        break

print("Images selected per class:")
for cid in range(NUM_CLASSES):
    print(f"  {CLASS_NAMES[cid]:>13}: {len(class_images[cid])}")

# ------------------------------------------------------------
# 2. Run inference on selected images
# ------------------------------------------------------------
all_img_paths = []
for cid in range(NUM_CLASSES):
    for img_path, _ in class_images[cid]:
        if img_path not in all_img_paths:
            all_img_paths.append(img_path)

print(f"\nRunning inference on {len(all_img_paths)} images...")
inference_results = model.predict(
    source=all_img_paths,
    imgsz=IMGSZ,
    device=DEVICE,
    conf=0.25,
    iou=0.5,
    max_det=MAX_DET,
    verbose=False,
)

# Build dict: img_path -> annotated image (BGR numpy array)
annotated = {}
for r, ipath in zip(inference_results, all_img_paths):
    annotated[os.path.basename(ipath)] = r.plot(conf=True, labels=True)

# ------------------------------------------------------------
# 3. Display 6x6 grid (12 classes x 3 samples = 36 images)
# ------------------------------------------------------------
N_ROWS = 6
N_COLS = 6
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(14, 14))
axes = axes.flatten()

idx = 0
for cid in range(NUM_CLASSES):
    for img_path, _ in class_images[cid]:
        if idx >= N_ROWS * N_COLS:
            break
        img_bgr = annotated.get(os.path.basename(img_path))
        if img_bgr is not None:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            axes[idx].imshow(img_rgb)
        axes[idx].set_title(CLASS_NAMES[cid], fontsize=7, fontweight='bold',
                            color=CLASS_COLORS[cid])
        axes[idx].axis('off')
        idx += 1

# Hide unused axes
for j in range(idx, len(axes)):
    axes[j].axis('off')

plt.tight_layout(pad=0.5)
plt.subplots_adjust(wspace=0.05, hspace=0.25)
plt.show()

# ------------------------------------------------------------
# 4. Correctness log -- per-class prediction accuracy
# ------------------------------------------------------------
def compute_iou_log(boxA, boxB):
    """Compute IoU between two boxes [x1, y1, x2, y2]."""
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    if interArea == 0:
        return 0.0
    boxAArea = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    boxBArea = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea)

print("\n" + "=" * 80)
print("  INFERENCE CORRECTNESS LOG -- Per-Image")
print("=" * 80)

class_correct = {i: 0 for i in range(NUM_CLASSES)}
class_missed  = {i: 0 for i in range(NUM_CLASSES)}
class_total   = {i: 0 for i in range(NUM_CLASSES)}

for cid in range(NUM_CLASSES):
    for img_path, lbl_path in class_images[cid]:
        class_total[cid] += 1
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        # Ground-truth class IDs for this image
        gt_classes_in_img = set()
        gt_boxes = []
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    gcls = int(float(parts[0]))
                    cx, cy, bw, bh = map(float, parts[1:5])
                    x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
                    x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
                    gt_classes_in_img.add(gcls)
                    gt_boxes.append((gcls, x1, y1, x2, y2))

        # Predicted class IDs + boxes for this image
        pred_classes_in_img = set()
        pred_boxes = []
        for r, ipath in zip(inference_results, all_img_paths):
            if os.path.basename(ipath) == os.path.basename(img_path):
                if r.boxes is not None:
                    for box_data in r.boxes.data.cpu().numpy():
                        px1, py1, px2, py2, conf, pcls = box_data
                        pred_classes_in_img.add(int(pcls))
                        pred_boxes.append((int(pcls), int(px1), int(py1), int(px2), int(py2)))
                break

        # Check: does any GT box of class cid match a pred of SAME class at IoU >= 0.5?
        found = False
        for gcls, gx1, gy1, gx2, gy2 in gt_boxes:
            if gcls != cid:
                continue
            for pcls, px1, py1, px2, py2 in pred_boxes:
                if pcls != cid:
                    continue
                iou = compute_iou_log([gx1, gy1, gx2, gy2], [px1, py1, px2, py2])
                if iou >= 0.5:
                    found = True
                    break
            if found:
                break

        short_name = os.path.basename(img_path)[:35]
        if found:
            class_correct[cid] += 1
            print(f"  [OK]  {CLASS_NAMES[cid]:>13} | {short_name:<35} | GT:{gt_classes_in_img}  Pred:{pred_classes_in_img}")
        else:
            class_missed[cid] += 1
            print(f"  [MISS] {CLASS_NAMES[cid]:>13} | {short_name:<35} | GT:{gt_classes_in_img}  Pred:{pred_classes_in_img}")

# ---- Summary table ----
print("\n" + "-" * 55)
header_cols = "Class           Correct  Missed   Accuracy"
print(f"  {header_cols}")
print("  " + "-" * 44)
for cid in range(NUM_CLASSES):
    total = class_total[cid]
    if total > 0:
        acc = class_correct[cid] / total * 100
        name = CLASS_NAMES[cid]
        print(f"  {name:<15} {class_correct[cid]:>7} {class_missed[cid]:>7} {acc:>9.1f}%")
    else:
        name = CLASS_NAMES[cid]
        print(f"  {name:<15}       0       0       N/A")

total_correct = sum(class_correct.values())
total_images  = sum(class_total.values())
print("  " + "-" * 44)
overall_label = "OVERALL"
print(f"  {overall_label:<15} {total_correct:>7} {total_images - total_correct:>7} {total_correct/total_images*100:>9.1f}%")
print("-" * 55)


# Error Analysis

Run inference on the entire test set and classify every prediction relative
to ground truth. Four error subtypes are tracked:

1. **False Negative (FN):** GT box with no matching prediction at IoU >= 0.5.
2. **False Positive -- Background:** predicted box with no GT overlap (IoU < 0.3).
3. **FP -- Misclassification:** IoU >= 0.5 to a GT box but wrong class.
4. **Localization Error (LocErr):** IoU 0.3--0.5 to matching-class GT.

Top 3 False Negatives (largest missed objects by area) and top 3 False
Positives (highest confidence) are visualised. Green = GT; red dashed =
predictions.

**For the viva:** compare this COCO-baseline failure mix with the SSL arms
(01b/02b/03b) -- the SSL backbone is expected to reduce pest-appearance
class confusion.

In [ ]:
model = YOLO(best_model_path)

# ---- Run inference on test set (save=False for speed) ----
test_img_dir = os.path.join(PARTB_ROOT, "test", "images")
test_lbl_dir = os.path.join(PARTB_ROOT, "test", "labels")
test_img_list = sorted([f for f in os.listdir(test_img_dir)
                        if f.lower().endswith(('.jpg','.jpeg','.png'))])

preds = model.predict(
    source=test_img_dir,
    imgsz=IMGSZ,
    device=DEVICE,
    conf=CONF,
    iou=IOU,
    max_det=MAX_DET,
    verbose=False,
    save=False,
)

# ---- IoU helper ----
def compute_iou(boxA, boxB):
    """box = [x1, y1, x2, y2] in absolute pixels."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    if interArea == 0:
        return 0.0
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea)


# ---- Classify errors: FN, FP-bg, FP-miscls, LocErr ----
fn_examples   = []  # (img_path, gt_box_abs, gt_class_name, fname)
fp_bg         = []  # (img_path, pred_box_abs, pred_cls_name, pred_conf, fname)
fp_miscls     = []  # (img_path, pred_box_abs, pred_cls_name, gt_cls_name, pred_conf, fname)
locerr        = []  # (img_path, pred_box_abs, pred_cls_name, gt_box_abs, iou, pred_conf, fname)

for pred_result, fname in zip(preds, test_img_list):
    img_path = os.path.join(test_img_dir, fname)
    lbl_path = os.path.join(test_lbl_dir, os.path.splitext(fname)[0] + '.txt')
    if not os.path.exists(lbl_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue
    h, w = img.shape[:2]

    # Parse ground truth
    gt_boxes = []  # [(cls, x1, y1, x2, y2)]
    with open(lbl_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(float(parts[0]))
            cx, cy, bw, bh = map(float, parts[1:5])
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            gt_boxes.append((cls, x1, y1, x2, y2))

    # Parse predictions
    pred_boxes = []  # [(cls, conf, x1, y1, x2, y2)]
    if pred_result.boxes is not None:
        for box_data in pred_result.boxes.data.cpu().numpy():
            x1, y1, x2, y2, conf, cls = box_data
            pred_boxes.append((int(cls), float(conf),
                               int(x1), int(y1), int(x2), int(y2)))

    # Greedy matching: pred -> GT, class-must-match for TP
    matched_gt   = set()
    matched_pred = set()
    best_match_for_pred = {}  # pi -> (gi, best_iou)

    pred_boxes_sorted = sorted(pred_boxes, key=lambda x: x[1], reverse=True)
    for pi, (pcls, pconf, px1, py1, px2, py2) in enumerate(pred_boxes_sorted):
        best_iou = 0.0
        best_gi  = -1
        best_gcls = -1
        for gi, (gcls, gx1, gy1, gx2, gy2) in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            iou = compute_iou([px1, py1, px2, py2], [gx1, gy1, gx2, gy2])
            if iou > best_iou:
                best_iou = iou
                best_gi  = gi
                best_gcls = gcls
        best_match_for_pred[pi] = (best_gi, best_iou, best_gcls)

        if best_iou >= 0.5 and best_gcls == pcls:
            # TP -- class matches
            matched_gt.add(best_gi)
            matched_pred.add(pi)
            
        elif best_iou >= 0.5 and best_gcls != pcls:
            # Misclassification -- right location, wrong class
            fp_miscls.append((img_path, (px1, py1, px2, py2),
                               CLASS_NAMES[pcls], CLASS_NAMES[best_gcls],
                               pconf, fname))
            matched_pred.add(pi)
            
        elif 0.3 <= best_iou < 0.5 and best_gcls == pcls:
            # Localization error -- right class, poor IoU
            # FIXED: Added 'gcls' to unpack all 5 elements
            gcls, gx1, gy1, gx2, gy2 = gt_boxes[best_gi]
            locerr.append((img_path, (px1, py1, px2, py2),
                            CLASS_NAMES[pcls], (gx1, gy1, gx2, gy2),
                            best_iou, pconf, fname))
            matched_pred.add(pi)

    # Unmatched GT boxes -> FN
    for gi, (gcls, gx1, gy1, gx2, gy2) in enumerate(gt_boxes):
        if gi not in matched_gt:
            fn_examples.append((img_path, (gx1, gy1, gx2, gy2),
                                CLASS_NAMES[gcls], fname))

    # Unmatched pred boxes -> FP (background)
    for pi, (pcls, pconf, px1, py1, px2, py2) in enumerate(pred_boxes_sorted):
        if pi not in matched_pred:
            fp_bg.append((img_path, (px1, py1, px2, py2),
                          CLASS_NAMES[pcls], pconf, fname))

# ---- Summary ----
print(f"\n{'='*55}")
print(f"  ERROR ANALYSIS SUMMARY")
print(f"  {'─'*40}")
print(f"  False Negatives (FN):                {len(fn_examples):>5}")
print(f"  False Positives -- Background:       {len(fp_bg):>5}")
print(f"  False Positives -- Misclassification: {len(fp_miscls):>5}")
print(f"  Localization Errors (LocErr):        {len(locerr):>5}")
print(f"{'='*55}")

# ---- Sort: FN by area (largest first), FP by confidence (highest first) ----
fn_examples.sort(key=lambda x: (x[1][2]-x[1][0])*(x[1][3]-x[1][1]), reverse=True)
fp_all = fp_bg + fp_miscls

# FIXED: Sorting by the second-to-last index to handle both 5-item and 6-item tuples
fp_all.sort(key=lambda x: x[-2], reverse=True)

# ---- Visualise ----
def draw_error_image(img_path, gt_box=None, gt_label=None,
                     pred_box=None, pred_label=None, pred_conf=None,
                     error_type="FN"):
    """Draw GT (green) and/or prediction (red dashed) on image."""
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(img)

    if gt_box is not None:
        x1, y1, x2, y2 = gt_box
        rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  fill=False, edgecolor='green', linewidth=2.5)
        ax.add_patch(rect)
        if gt_label:
            ax.text(x1, max(y1-4, 6), f"GT: {gt_label}",
                    fontsize=8, color='white', weight='bold',
                    bbox=dict(boxstyle='round,pad=0.1',
                              facecolor='green', alpha=0.8))

    if pred_box is not None:
        x1p, y1p, x2p, y2p = pred_box
        rect_p = mpatches.Rectangle((x1p, y1p), x2p-x1p, y2p-y1p,
                                     fill=False, edgecolor='red',
                                     linewidth=2.5, linestyle='--')
        ax.add_patch(rect_p)
        if pred_label:
            lbl = f"Pred: {pred_label}"
            if pred_conf is not None:
                lbl += f" ({pred_conf:.2f})"
            ax.text(x1p, max(y2p+10, 6), lbl,
                    fontsize=8, color='white', weight='bold',
                    bbox=dict(boxstyle='round,pad=0.1',
                              facecolor='red', alpha=0.8))

    ax.set_title(f"{error_type} -- {os.path.basename(img_path)[:35]}", fontsize=9)
    ax.axis('off')
    return fig


TOP_N = 3

# ---- False Negatives ----
print(f"\n{'─'*55}")
print(f"  TOP {TOP_N} FALSE NEGATIVES (largest missed objects by area)")
print(f"{'─'*55}")
for i in range(min(TOP_N, len(fn_examples))):
    img_path, gt_box, gt_cls_name, fname = fn_examples[i]
    print(f"\n  FN {i+1}: {fname[:45]} -- GT: {gt_cls_name}")
    fig = draw_error_image(img_path, gt_box=gt_box, gt_label=gt_cls_name,
                           error_type=f"FN #{i+1}")
    plt.show()
    plt.close(fig)

# ---- False Positives (combined background + misclassification) ----
print(f"\n{'─'*55}")
print(f"  TOP {TOP_N} FALSE POSITIVES (highest confidence)")
print(f"{'─'*55}")
for i in range(min(TOP_N, len(fp_all))):
    if len(fp_all[i]) == 6:
        # FP-miscls
        img_path, pred_box, pred_cls, gt_cls, pred_conf, fname = fp_all[i]
        print(f"\n  FP {i+1}: {fname[:45]} -- Pred: {pred_cls} (conf={pred_conf:.3f})")
        print(f"         GT class: {gt_cls} -- MISCLASSIFICATION")
    else:
        # FP-bg
        img_path, pred_box, pred_cls, pred_conf, fname = fp_all[i]
        print(f"\n  FP {i+1}: {fname[:45]} -- Pred: {pred_cls} (conf={pred_conf:.3f})")
        print(f"         BACKGROUND -- no GT match")
    fig = draw_error_image(img_path, pred_box=pred_box,
                           pred_label=pred_cls, pred_conf=pred_conf,
                           error_type=f"FP #{i+1}")
    plt.show()
    plt.close(fig)

# Save Checkpoint

Copy the best checkpoint to `/kaggle/working/` for download or cross-model
comparison with the SSL-initialised arms.

In [ ]:
import shutil

src = os.path.join(PROJECT, RUN_NAME, "weights", "best.pt")
dst = os.path.join("/kaggle/working", "yolov12_baseline_coco20_best.pt")

if os.path.exists(src):
    shutil.copy(src, dst)
    size_mb = os.path.getsize(dst) / (1024**2)
    print(f"Checkpoint saved: {dst}")
    print(f"  Size: {size_mb:.1f} MB")
else:
    print(f"Source not found: {src}")